# CfD Hedge Model — Part 3: Streamlit Dashboard

This notebook writes the Streamlit app file and shows you how to run it.
The dashboard is the deliverable you link in your application.

**To run:** After running the cell below, open a terminal and run:
```
streamlit run cfd_dashboard.py
```


In [8]:

# Write the Streamlit app to disk
# This is the complete dashboard code

app_code = """
import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from dataclasses import dataclass
from typing import List

st.set_page_config(page_title="CfD Hedge Model", page_icon="⚡", layout="wide")

# ── Model classes ────────────────────────────────────────────────────

@dataclass
class CfDPosition:
    name: str
    strike_price: float
    volume_mwh: float
    direction: str

    def settlement(self, reference_price: float) -> float:
        diff = reference_price - self.strike_price
        payout = diff * self.volume_mwh
        return payout if self.direction == "long" else -payout

def temperature_to_demand(temp_c, base_load, hdd_base=15.5, sensitivity=80):
    hdd = max(hdd_base - temp_c, 0)
    base_hdd = max(hdd_base - 10, 0)
    return base_load + (hdd - base_hdd) * sensitivity

def weather_to_price(temp_c, wind_cf, base_price, base_load):
    demand = temperature_to_demand(temp_c, base_load)
    wind_gen = 6000 * np.clip(wind_cf, 0, 1)
    demand_effect = (demand - base_load) * 0.003
    wind_effect = (wind_gen - 0.35 * 6000) * -0.005
    return round(base_price + demand_effect + wind_effect, 2)

def run_monte_carlo(portfolio, base_load, base_price, n=1000, seed=42):
    np.random.seed(seed)
    temps = np.random.normal(7.0, 5.0, n)
    wind_cfs = np.clip(np.random.normal(0.35, 0.12, n), 0.05, 0.80)
    results = []
    for t, w in zip(temps, wind_cfs):
        ref = weather_to_price(t, w, base_price, base_load)
        demand = temperature_to_demand(t, base_load)
        hedged_vol = sum(p.volume_mwh for p in portfolio)
        cfd_net = sum(p.settlement(ref) for p in portfolio)
        unhedged_cost = max(demand - hedged_vol, 0) * ref
        results.append({"temp": t, "wind_cf": w, "ref_price": ref,
                         "demand": demand, "cfd_settlement": cfd_net,
                         "net_cost": unhedged_cost - cfd_net})
    return pd.DataFrame(results)

# ── Sidebar controls ─────────────────────────────────────────────────

st.sidebar.title("Model inputs")

base_price = st.sidebar.slider("Base reference price (£/MWh)", 30, 130, 70)
base_load = st.sidebar.slider("Total load exposure (MWh)", 5000, 30000, 16000, step=500)

st.sidebar.markdown("---")
st.sidebar.markdown("**CfD portfolio**")
wind_a_vol = st.sidebar.slider("Wind CfD A volume (MWh)", 0, 10000, 5000, step=500)
wind_a_strike = st.sidebar.slider("Wind CfD A strike (£/MWh)", 30, 100, 55)
wind_b_vol = st.sidebar.slider("Wind CfD B volume (MWh)", 0, 10000, 3000, step=500)
solar_vol = st.sidebar.slider("Solar CfD volume (MWh)", 0, 10000, 2000, step=500)
gas_vol = st.sidebar.slider("Gas short volume (MWh)", 0, 10000, 4000, step=500)

st.sidebar.markdown("---")
st.sidebar.markdown("**Weather scenario**")
temp_c = st.sidebar.slider("Temperature (°C)", -10, 20, 10)
wind_cf = st.sidebar.slider("Wind capacity factor", 0.05, 0.80, 0.35, step=0.01)

# ── Build portfolio ───────────────────────────────────────────────────

portfolio = [
    CfDPosition("Wind CfD A",     wind_a_strike, wind_a_vol, "long"),
    CfDPosition("Wind CfD B",     62,            wind_b_vol, "long"),
    CfDPosition("Solar CfD",      48,            solar_vol,  "long"),
    CfDPosition("Gas short",      80,            gas_vol,    "short"),
]

ref_price    = weather_to_price(temp_c, wind_cf, base_price, base_load)
demand       = temperature_to_demand(temp_c, base_load)
hedged_vol   = sum(p.volume_mwh for p in portfolio)
hedge_ratio  = hedged_vol / base_load
cfd_net      = sum(p.settlement(ref_price) for p in portfolio)
unhedged     = max(demand - hedged_vol, 0)
net_cost     = unhedged * ref_price - cfd_net

# ── Header ────────────────────────────────────────────────────────────

st.title("CfD Hedge Position Model")
st.caption("Migrated from Excel · Python · E.ON internship project")

col1, col2, col3, col4 = st.columns(4)
col1.metric("Reference price",  f"£{ref_price:.1f}/MWh",
            delta=f"{ref_price - base_price:+.1f} vs base")
col2.metric("Hedge ratio",       f"{hedge_ratio:.1%}",
            delta="Target 85%" if hedge_ratio < 0.85 else "On target")
col3.metric("CfD settlement",   f"£{cfd_net/1000:,.0f}k")
col4.metric("Net cost",          f"£{net_cost/1000:,.0f}k",
            delta_color="inverse")

# Hedge instruction
if hedge_ratio < 0.70:
    st.warning(f"Hedge ratio below 70%. Consider adding ~{(base_load*0.85 - hedged_vol):,.0f} MWh volume.")
elif hedge_ratio > 1.05:
    st.warning(f"Over-hedged by {(hedged_vol - base_load):,.0f} MWh. Consider unwinding excess positions.")
else:
    st.success("Hedge ratio within target range (70–105%). No action required.")

st.markdown("---")

# ── Charts ────────────────────────────────────────────────────────────

col_l, col_r = st.columns(2)

with col_l:
    st.subheader("P&L vs reference price")
    prices = np.linspace(-20, 150, 300)
    net_vals = [sum(p.settlement(px) for p in portfolio) / 1000 for px in prices]
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=prices, y=net_vals, mode="lines",
                              line=dict(color="#185FA5", width=2), name="Net settlement"))
    fig1.add_vline(x=ref_price, line_dash="dash", line_color="#1D9E75",
                   annotation_text=f"Current £{ref_price}")
    fig1.add_vline(x=0, line_dash="dot", line_color="#D85A30",
                   annotation_text="Negative price")
    fig1.update_layout(xaxis_title="Reference price (£/MWh)",
                        yaxis_title="Net settlement (£000s)",
                        height=350, margin=dict(t=20))
    st.plotly_chart(fig1, use_container_width=True)

with col_r:
    st.subheader("Settlement by position")
    pos_data = [{"name": p.name,
                 "settlement_k": round(p.settlement(ref_price) / 1000, 1)}
                for p in portfolio]
    pos_df = pd.DataFrame(pos_data)
    colours = ["#1D9E75" if v >= 0 else "#D85A30" for v in pos_df["settlement_k"]]
    fig2 = go.Figure(go.Bar(x=pos_df["name"], y=pos_df["settlement_k"],
                             marker_color=colours, text=pos_df["settlement_k"].apply(lambda x: f"£{x:,.0f}k"),
                             textposition="outside"))
    fig2.update_layout(yaxis_title="Settlement (£000s)", height=350, margin=dict(t=20))
    st.plotly_chart(fig2, use_container_width=True)

# ── Monte Carlo ───────────────────────────────────────────────────────

st.subheader("Monte Carlo weather simulation (1,000 scenarios)")

mc_df = run_monte_carlo(portfolio, base_load, base_price)

var95 = np.percentile(mc_df["net_cost"], 95)
var99 = np.percentile(mc_df["net_cost"], 99)
mean_c = mc_df["net_cost"].mean()

mc1, mc2, mc3 = st.columns(3)
mc1.metric("Mean net cost",  f"£{mean_c/1000:,.0f}k")
mc2.metric("VaR 95%",        f"£{var95/1000:,.0f}k")
mc3.metric("VaR 99%",        f"£{var99/1000:,.0f}k")

fig3 = px.histogram(mc_df, x=mc_df["net_cost"]/1e6, nbins=50,
                    labels={"x": "Net cost (£M)"},
                    color_discrete_sequence=["#378ADD"])
fig3.add_vline(x=var95/1e6, line_dash="dash", line_color="#D85A30",
               annotation_text=f"VaR 95%")
fig3.add_vline(x=mean_c/1e6, line_dash="dash", line_color="#1D9E75",
               annotation_text="Mean")
fig3.update_layout(height=320, margin=dict(t=10))
st.plotly_chart(fig3, use_container_width=True)

# ── Raw data table ────────────────────────────────────────────────────

with st.expander("Show position detail table"):
    detail = pd.DataFrame([{
        "Position": p.name,
        "Direction": p.direction,
        "Strike (£/MWh)": p.strike_price,
        "Volume (MWh)": f"{p.volume_mwh:,.0f}",
        "Settlement (£)": f"{p.settlement(ref_price):,.0f}"
    } for p in portfolio])
    st.dataframe(detail, use_container_width=True)
"""


with open('cfd_dashboard.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('cfd_dashboard.py written successfully.')
print()
print('To run locally:    streamlit run cfd_dashboard.py')


cfd_dashboard.py written successfully.

To run locally:    streamlit run cfd_dashboard.py


## GitHub README template

Copy this as your `README.md`:

```markdown
# CfD Hedge Position Model

A Python replication and extension of a Contracts for Difference (CfD) hedge position model — 
migrated from Excel to a modular, testable codebase with interactive dashboard.

## What it does
- Models a portfolio of CfD positions (long/short, various tenors and strikes)
- Calculates settlement including negative price scenarios (relevant for UK renewable CfDs)
- Links weather inputs (temperature, wind capacity factor) to electricity demand and price
- Monte Carlo simulation over 1,000 weather outcomes → VaR-style risk metrics
- Streamlit dashboard with interactive sliders for live scenario analysis

## Run it
```bash
pip install streamlit plotly pandas numpy
streamlit run cfd_dashboard.py
```

## Extensions planned
- Seasonal tenor modelling (summer vs winter shape)
- Clean Power 2030 scenario: higher wind penetration → more frequent negative prices
- CfD derivative valuation (Black-Scholes for energy options)
- Integration with live Elexon/BMRS price data via API
```